In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler,StandardScaler,OneHotEncoder
from category_encoders.count import CountEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
data_path = root / 'data/raw'
song_data_path = data_path / 'Music Info.csv'
user_data_path = root / 'data/local/listens.csv'

In [ ]:
# load the song data

df_songs = pd.read_csv(song_data_path)

df_songs.sample(3)

In [ ]:
# duplicates in dataset

df_songs.duplicated().sum()

In [ ]:
# duplicates based on spotify_id

df_songs.duplicated(subset=["spotify_id","year","duration_ms"]).sum()

In [ ]:
# drop duplicated rows

df_songs.drop_duplicates(subset=["spotify_id","year","duration_ms"],inplace=True)

df_songs.duplicated(subset=["spotify_id","year","duration_ms"]).sum()

In [ ]:
df_songs.reset_index(drop=True,inplace=True)

In [ ]:
# remove columns not required for content filtering

columns_to_remove = ["track_id","name","spotify_preview_url","spotify_id","genre"]

df_content_filtering = df_songs.drop(columns=columns_to_remove)

df_content_filtering.sample(3)

In [ ]:
# check for missing values

df_content_filtering.isnull().sum()

In [ ]:
# fill the tags column missing values with string "no_tags"

df_content_filtering.fillna({'tags':'no_tags'},inplace=True)

In [ ]:
df_content_filtering.isnull().sum()

In [ ]:
# number of unique artist

df_content_filtering['artist'].str.lower().nunique()

In [ ]:
# number of unique year values

df_content_filtering.loc[:,'year'].nunique()

In [ ]:
# min and max values of numerical columns

df_content_filtering.select_dtypes(include='number').describe().loc[['min','max']]

In [ ]:
(
    df_content_filtering
    .loc[:,'tags']
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
)

In [ ]:
(
    df_content_filtering
    .loc[:,'tags']
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .loc[lambda x : x>1000]
)

In [ ]:
# columns to transform

frequency_encode_cols = ['year']
ohe_cols = ['artist','time_signature','key']
tfidf_col = ['tags']
standard_scaler_cols = ['duration_ms','loudness','tempo']
min_max_scaler_cols = ['danceability','energy','speechiness','acousticness','instrumentalness','liveness','valence']


In [ ]:
# transform the data

transformer = ColumnTransformer(
    transformers=[
        ('frequency_encode',CountEncoder(normalize=True,return_df=True),frequency_encode_cols),
        ('ohe',OneHotEncoder(handle_unknown='ignore'),ohe_cols),
        ('tfidf',TfidfVectorizer(max_features=85),tfidf_col[0]),
        ('standard_scale',StandardScaler(),standard_scaler_cols),
        ('min_max_scale',MinMaxScaler(),min_max_scaler_cols)
    ],
    remainder='passthrough',n_jobs=1,verbose_feature_names_out=False
)

transformer

In [ ]:
# fit the transformer

transformer.fit(df_content_filtering)

In [ ]:
transformed_df = transformer.transform(df_content_filtering)

In [ ]:
transformed_df.shape

In [ ]:
transformed_df

In [ ]:
df_content_filtering.shape

In [ ]:
# build input vector

song_input = df_content_filtering[df_songs['name'] == "Whenever, Wherever"]

song_input

In [ ]:
# input vector to calculate similarity

input_vector = transformer.transform(song_input)

input_vector

In [ ]:
input_vector.shape

In [ ]:
similarity_score = cosine_similarity(transformed_df,input_vector)

similarity_score

In [ ]:
similarity_score.shape

In [ ]:
top_10_songs_indexes = np.argsort(similarity_score,axis=0).ravel()[-11:][::-1]

top_10_songs_indexes

In [ ]:
top_10_songs_names = df_songs.iloc[top_10_songs_indexes]

top_10_songs_names

In [ ]:
def recommend(song_name, songs_data, transformed_data, k=10):
    
    """
    Recommends top k songs similar to the given song based on content-based filtering.

    Parameters:
    song_name (str): The name of the song to base the recommendations on.
    songs_data (DataFrame): The DataFrame containing song information.
    transformed_data (ndarray): The transformed data matrix for similarity calculations.
    k (int, optional): The number of similar songs to recommend. Default is 10.

    Returns:
    DataFrame: A DataFrame containing the top k recommended songs with their names, artists, and Spotify preview URLs.
    """

    # filter out the song from data
    song_row = songs_data[songs_data['name'] == song_name]

    if song_row.empty:
        print('Song not found in the dataset')
    else:
        # get the index of the song
        song_index = song_row.index[0]
        print(f'Index of the song entered is {song_index}')

        # generate the input vector
        input_vector = transformed_data[song_index]

        # calculate similarity score
        similarity_score = cosine_similarity(input_vector,transformed_data)
        print(f'Similarity score shape is : {similarity_score.shape}')

        # get the top k songs
        top_k_songs_indexes = np.argsort(similarity_score,axis=1).ravel()[-k-1:][::-1]
        print(f"Top {k} songs indexes are {top_k_songs_indexes}")

        # get the top k songs names
        top_k_songs_names = songs_data.iloc[top_k_songs_indexes]

        # print the top k songs
        top_k_list = top_10_songs_names[['name','artist','spotify_preview_url']].reset_index(drop=True)

        return top_k_list

In [ ]:
recommend("Whenever, Wherever",songs_data=df_songs,transformed_data=transformed_df,k=10)